In [0]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
 
CATALOG    = "clutchlytics"
SOURCE     = f"{CATALOG}.silver.nhl_series"
GOLD_TABLE = f"{CATALOG}.gold.nhl_gold_series_momentum"
 
LEAGUE = "nhl"
SPORT  = "hockey"
 
print(f"Source : {SOURCE}")
print(f"Target : {GOLD_TABLE}")

In [0]:
# ── READ SOURCE ───────────────────────────────────────────────────────────────
 
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone
 
series_df = spark.table(SOURCE).filter(F.col("league") == LEAGUE)
print(f"nhl_series rows : {series_df.count()}")
print(f"Unique series   : {series_df.select('series_key').distinct().count()}")

In [0]:
# ── WINDOW SPECS ──────────────────────────────────────────────────────────────
 
series_window_ordered = Window.partitionBy("series_key").orderBy("game_number")
series_window_all     = Window.partitionBy("series_key")

In [0]:
# ── SERIES-LEVEL AGGREGATIONS ─────────────────────────────────────────────────
# Computed once per series, broadcast across all game rows via window functions.
 
# ── Total games played in series ──
series_df = series_df.withColumn(
    "series_total_games_played",
    F.max("game_number").over(series_window_all)
)
 
# ── Series winner — team with 4 wins, broadcast to all rows ──
series_df = series_df.withColumn(
    "series_winner_abbr",
    F.when(F.col("team_a_wins") == 4, F.col("team_a_abbr"))
     .when(F.col("team_b_wins") == 4, F.col("team_b_abbr"))
     .otherwise(None)
)
 
series_df = series_df.withColumn(
    "series_winner_abbr",
    F.last("series_winner_abbr", ignorenulls=True).over(series_window_all)
)
 
# ── Game 1 winner — broadcast to all rows ──
series_df = series_df.withColumn(
    "game_1_winner_abbr",
    F.first(
        F.when(F.col("game_number") == 1, F.col("game_winner_abbr")),
        ignorenulls=True
    ).over(series_window_all)
)
 
# ── Game 1 winner won series — only populated on clinching game row ──
series_df = series_df.withColumn(
    "game_1_winner_won_series",
    F.when(
        F.col("series_clinched") == True,
        F.col("series_winner_abbr") == F.col("game_1_winner_abbr")
    ).otherwise(None)
)
 
# ── Largest series lead — max win differential at any point ──
series_df = series_df.withColumn(
    "largest_lead",
    F.max(F.abs(F.col("team_a_wins") - F.col("team_b_wins")))
     .over(series_window_all)
)

In [0]:
# ── came_from_behind DETECTION ────────────────────────────────────────────────
# Strict definition: team trailed 3-1 AND won the series.
# Flag the series if at any game row one team had 1 win while the other had 3.
 
# Step 1 — flag any game where one team led 3-1
series_df = series_df.withColumn(
    "was_31_deficit_game",
    F.when(
        ((F.col("team_a_wins") == 3) & (F.col("team_b_wins") == 1)) |
        ((F.col("team_b_wins") == 3) & (F.col("team_a_wins") == 1)),
        True
    ).otherwise(False)
)
 
# Step 2 — broadcast whether ANY game in the series had a 3-1 deficit
series_df = series_df.withColumn(
    "series_had_31_deficit",
    F.max(F.col("was_31_deficit_game").cast("integer")).over(series_window_all)
    .cast("boolean")
)
 
# Step 3 — who was trailing 3-1 (the team with 1 win when deficit existed)
series_df = series_df.withColumn(
    "team_trailing_31",
    F.when(
        F.col("was_31_deficit_game") == True,
        F.when(F.col("team_a_wins") == 1, F.col("team_a_abbr"))
         .when(F.col("team_b_wins") == 1, F.col("team_b_abbr"))
    ).otherwise(None)
)
 
series_df = series_df.withColumn(
    "team_trailing_31",
    F.last("team_trailing_31", ignorenulls=True).over(series_window_all)
)
 
# Step 4 — came_from_behind = trailing team won the series
series_df = series_df.withColumn(
    "came_from_behind",
    F.when(
        F.col("series_had_31_deficit") == True,
        F.col("team_trailing_31") == F.col("series_winner_abbr")
    ).otherwise(False)
)

In [0]:
# ── series_competitiveness DERIVATION ─────────────────────────────────────────
# sweep        → 4 games
# dominant     → 5 games
# competitive  → 6 or 7 games, no 3-1 comeback
# comeback     → team trailed 3-1 and won
 
series_df = series_df.withColumn(
    "series_competitiveness",
    F.when(F.col("came_from_behind") == True,                         "comeback")
     .when(F.col("series_total_games_played") == 4,                   "sweep")
     .when(F.col("series_total_games_played") == 5,                   "dominant")
     .when(F.col("series_total_games_played").isin(6, 7),             "competitive")
     .otherwise("in_progress")
)

In [0]:
# ── FINAL COLUMN SELECTION ────────────────────────────────────────────────────
 
ingested_at = datetime.now(timezone.utc).isoformat()
 
gold_df = series_df.select(
    # ── Series identity ──
    "series_key",
    "round",
    "season",
    F.lit(SPORT).alias("sport"),
    F.lit(LEAGUE).alias("league"),
    "season_type",
 
    # ── Teams ──
    "team_a_clutch_id",
    "team_b_clutch_id",
    "team_a_abbr",
    "team_b_abbr",
 
    # ── Per game ──
    "game_number",
    "clutch_game_id",
    "source_event_id",
 
    # ── Game result ──
    "home_team_abbr",
    "away_team_abbr",
    "home_score",
    "away_score",
    "went_to_ot",
    "goal_differential",
    "cumulative_goal_diff",
    "game_winner_abbr",
 
    # ── Series standing after this game ──
    "team_a_wins",
    "team_b_wins",
    "series_leader_abbr",
    "series_tied",
    "series_clinched",
 
    # ── Series-level Gold derivations ──
    "series_total_games_played",
    "series_winner_abbr",
    "game_1_winner_abbr",
    "game_1_winner_won_series",
    "largest_lead",
    "came_from_behind",
    "series_competitiveness",
 
    # ── Metadata ──
    F.lit(ingested_at).alias("ingested_at"),
    F.lit("silver.nhl_series").alias("source_table"),
)
 
print(f"Total rows to write: {gold_df.count()}")

In [0]:
# ── PREVIEW — series competitiveness summary ──────────────────────────────────
 
print("── Series competitiveness breakdown ──")
gold_df.filter(F.col("series_clinched") == True).select(
    "series_key",
    "team_a_abbr",
    "team_b_abbr",
    "series_winner_abbr",
    "series_total_games_played",
    "series_competitiveness",
    "came_from_behind",
    "game_1_winner_abbr",
    "game_1_winner_won_series",
    "largest_lead",
).orderBy("round", "series_competitiveness").show(20, truncate=False)

In [0]:
# ── WRITE TO GOLD ─────────────────────────────────────────────────────────────
# Full refresh on each run — Gold is derived, always reproducible from Silver.
 
(
    gold_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)
 
print(f"Written to {GOLD_TABLE}")

In [0]:
# ── VALIDATE ─────────────────────────────────────────────────────────────────
 
print("── Game 1 winner analysis ──")
spark.sql(f"""
    SELECT
        series_key,
        game_1_winner_abbr,
        series_winner_abbr,
        game_1_winner_won_series,
        series_total_games_played,
        series_competitiveness
    FROM {GOLD_TABLE}
    WHERE series_clinched = true
    ORDER BY series_key
""").show(truncate=False)
 
print("── Competitiveness distribution ──")
spark.sql(f"""
    SELECT
        series_competitiveness,
        COUNT(DISTINCT series_key) AS series_count
    FROM {GOLD_TABLE}
    GROUP BY series_competitiveness
    ORDER BY series_count DESC
""").show(truncate=False)
 

In [0]:
# ── SANITY CHECKS ─────────────────────────────────────────────────────────────
 
checks = spark.sql(f"""
    SELECT
        COUNT(*)                                                    AS total_rows,
        COUNT(DISTINCT series_key)                                  AS unique_series,
        COUNT(DISTINCT series_competitiveness)                      AS competitiveness_categories,
        COUNT(CASE WHEN series_competitiveness = 'sweep'       THEN series_key END)
            / COUNT(DISTINCT series_key) * 100                     AS pct_sweeps,
        COUNT(CASE WHEN came_from_behind = true THEN 1 END)        AS comeback_games,
        COUNT(CASE WHEN game_1_winner_won_series = true
                    AND series_clinched = true THEN 1 END)         AS g1_winner_won,
        COUNT(CASE WHEN game_1_winner_won_series = false
                    AND series_clinched = true THEN 1 END)         AS g1_winner_lost,
        COUNT(CASE WHEN series_winner_abbr IS NULL THEN 1 END)     AS null_series_winners,
        COUNT(CASE WHEN series_competitiveness = 'in_progress'
                    THEN 1 END)                                     AS in_progress_rows
    FROM {GOLD_TABLE}
""")
 
print("Sanity checks:")
checks.show(truncate=False)